# Green NLP Benchmark Lab - Notebook 02a: Quantization — coding workshop (advanced)

**Session 2 of 3.** Session 1 built a baseline QA and MT pipeline and measured its cost. This notebook reruns the same tasks with a quantized model configuration.

Expected time: **90-125 minutes**, depending on Colab installation time and GPU availability.

What should stay the same: corpus, questions, prompts, scoring rule, and measurement procedure.

What changes: numerical precision and loading path. The notebook tries GPTQ Int4 first; if that fails, it falls back to bitsandbytes Int8. Record which path loaded before interpreting the results.

**What you will learn:**

- to load a quantized version of the Session 1 model (GPTQ, 4 bits; or bitsandbytes, 8 bits, when GPTQ does not load)
  and to note which one you got;
- to rerun the Session 1 quiz with only the precision changed, and measure its energy and emissions with CodeCarbon, its time,
  and its peak GPU memory;
- to compare today's accuracy and cost with your Session 1 baseline row in `leaderboard.csv`;
- to translate our Galician passage with the quantized model and score the output against the reference
  translation, the gold output, with BLEU, BERTScore and two LLM judges.


## How this workshop works

This is the coding version of the notebook, **advanced level**. In some code cells, lines
are replaced by `# TODO` comments and `___`. Each exercise above such a cell says what the
code must do; the comments in the cell give the details. This version takes longer than
the expected time given above.

1. Replace every `___` with your code, then run the cell. Open the hints if you need them.
2. Run the check cell right below it. It prints `✓ Exercise N passed`, or an error
   message that says what to look at.
3. If you are stuck, the solution is below the check cell. Open it after you have tried.
   Or tick **fill_in_solutions** in the cell right below these instructions and run that cell:
   from then on, every exercise cell you run gets its solution in place of its code. With the box
   ticked, Runtime → Run all runs the whole notebook with the solutions.

A cell that still contains `___` stops with `ExerciseGaps: this cell still has gaps marked ___`.
Later cells use your code, so finish each exercise before moving on.

Exercises marked **(advanced)** are only in this version. In them you implement an
equation from the lesson or a paper, or a technique that makes the pipeline cheaper to
run, and check it against the library or the baseline.

In [ ]:
# @title Run this cell first. Tick the box to fill in the solutions (point 3 above).
fill_in_solutions = False  # @param {type:"boolean"}

import io
import re
import tokenize

_solutions = {
    1: "def tensor_bytes(m):\n    tensors = list(m.parameters()) + list(m.buffers())\n    return sum(t.numel() * t.element_size() for t in tensors)\n\n\ndef formula_gb(n_params, bits):\n    return n_params * bits / 8 / 1e9\n\nmeasured_gb = tensor_bytes(model) / 1e9\nprint(f\"Loaded configuration: {loaded_config}\")\nprint(f\"Bytes in the model's tensors: {measured_gb:.2f} GB\")\nprint(f\"Formula, 1.54B parameters at 16 bits: {formula_gb(1.54e9, 16):.2f} GB\")\nprint(f\"Formula, 1.54B parameters at  4 bits: {formula_gb(1.54e9, 4):.2f} GB\")\nprint(f\"Formula, 1.54B parameters at  8 bits: {formula_gb(1.54e9, 8):.2f} GB\")\n\nby_dtype = {}\nfor t in list(model.parameters()) + list(model.buffers()):\n    by_dtype[str(t.dtype)] = by_dtype.get(str(t.dtype), 0) + t.numel() * t.element_size()\nfor dtype, n in sorted(by_dtype.items(), key=lambda kv: -kv[1]):\n    print(f\"  {dtype:15s} {n / 1e9:.3f} GB\")",
    2: "def build_prompt(question, corpus):\n    context = \"\\n\\n\".join(f\"[{name}]\\n{text.strip()}\" for name, text in corpus.items())\n    messages = [\n        {\n            \"role\": \"system\",\n            \"content\": (\n                \"You are a concise assistant. Answer the question using ONLY the \"\n                \"documents provided below. Answer in one short sentence.\"\n            ),\n        },\n        {\n            \"role\": \"user\",\n            \"content\": f\"Documents:\\n{context}\\n\\nQuestion: {question}\",\n        },\n    ]\n    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n\n\ndef answer_question(question, corpus, max_new_tokens=60):\n    prompt = build_prompt(question, corpus)\n    inputs = tokenizer(prompt, return_tensors=\"pt\").to(model.device)\n    with torch.no_grad():\n        output = model.generate(\n            **inputs,\n            max_new_tokens=max_new_tokens,\n            do_sample=False,\n        )\n    generated = output[0][inputs[\"input_ids\"].shape[1]:]\n    return tokenizer.decode(generated, skip_special_tokens=True).strip()\n\n\nsample_answer = answer_question(QA_PAIRS[0][\"question\"], CORPUS)\nprint(\"Q:\", QA_PAIRS[0][\"question\"])\nprint(\"A:\", sample_answer)",
    3: "import os\n\nenergy_kwh = get_tracker_energy_kwh(tracker)\n\nmem = gpu_memory_summary()\n\nleaderboard_row = {\n    \"config\": loaded_config,\n    \"model\": loaded_model_name,\n    \"params_billion\": 1.5,\n    \"n_questions\": len(QA_PAIRS),\n    \"duration_sec\": round(elapsed, 2),\n    \"load_seconds\": round(load_seconds, 2),\n    \"energy_kwh\": energy_kwh,\n    \"emissions_kg_co2eq\": emissions_kg,\n    \"accuracy\": accuracy,\n    \"peak_gpu_memory_gb\": mem[\"peak_gpu_memory_gb\"],\n}\n\nif os.path.exists(\"leaderboard.csv\"):\n    leaderboard = pd.read_csv(\"leaderboard.csv\")\n    leaderboard = pd.concat([leaderboard, pd.DataFrame([leaderboard_row])], ignore_index=True)\n    print(f\"Appended to existing leaderboard.csv ({len(leaderboard) - 1} prior row(s)).\")\nelse:\n    leaderboard = pd.DataFrame([leaderboard_row])\n    print(\"No existing leaderboard.csv found - starting a fresh one.\")\n\nleaderboard.to_csv(\"leaderboard.csv\", index=False)\nleaderboard",
    4: "def count_tokens(text):\n    return len(tokenizer(text)[\"input_ids\"])\n\n\ndef joules(kwh):\n    return kwh * 3.6e6\n\n\ndef joules_per_correct(total_joules, results_list):\n    n_correct = sum(r[\"correct\"] for r in results_list)\n    return total_joules / n_correct if n_correct else float(\"inf\")\n\nn_generated = sum(count_tokens(r[\"answer\"]) for r in results)\ntoday = joules(energy_kwh)\nprint(f\"Today ({loaded_config}): {today:.1f} J for {n_generated} generated tokens\")\nprint(f\"  {today / n_generated:.3f} J per generated token\")\nprint(f\"  {today / len(results):.1f} J per question\")\nprint(f\"  {joules_per_correct(today, results):.1f} J per correct answer\")\n\nbaseline_rows = leaderboard[leaderboard[\"config\"] == \"baseline-fp16\"]\nif len(baseline_rows) > 0 and pd.notna(baseline_rows.iloc[-1][\"energy_kwh\"]):\n    b = baseline_rows.iloc[-1]\n    b_joules = joules(b[\"energy_kwh\"])\n    print(f\"Baseline-fp16 (Session 1): {b_joules / b['n_questions']:.1f} J per question, \"\n          f\"{b_joules / max(b['accuracy'] * b['n_questions'], 1e-12):.1f} J per correct answer\")",
    5: "import matplotlib.pyplot as plt\n\nbaseline_rows = leaderboard[leaderboard[\"config\"] == \"baseline-fp16\"]\n\nif len(baseline_rows) > 0:\n    baseline = baseline_rows.iloc[-1]\n    today = leaderboard.iloc[-1]\n\n    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))\n\n    configs = [baseline[\"config\"], today[\"config\"]]\n    energies = [baseline[\"emissions_kg_co2eq\"] * 1000, today[\"emissions_kg_co2eq\"] * 1000]\n    accuracies = [baseline[\"accuracy\"] * 100, today[\"accuracy\"] * 100]\n\n    axes[0].bar(configs, energies)\n    axes[0].set_ylabel(\"g CO2eq\")\n    axes[0].set_title(\"Emissions\")\n    axes[0].tick_params(axis=\"x\", rotation=15)\n\n    axes[1].bar(configs, accuracies)\n    axes[1].set_ylabel(\"accuracy (%)\")\n    axes[1].set_title(\"Accuracy\")\n    axes[1].set_ylim(0, 100)\n    axes[1].tick_params(axis=\"x\", rotation=15)\n\n    plt.tight_layout()\n    plt.show()\n\n    ratio = baseline[\"emissions_kg_co2eq\"] / max(today[\"emissions_kg_co2eq\"], 1e-12)\n    print(f\"baseline-fp16 emitted {ratio:.2f}x as much CO2eq as today's run \"\n          f\"(>1 means today's run was cheaper).\")\n    print(f\"Accuracy: baseline {baseline['accuracy']:.0%} vs. today {today['accuracy']:.0%}.\")\n    if \"peak_gpu_memory_gb\" in today and pd.notna(today[\"peak_gpu_memory_gb\"]):\n        print(f\"Peak GPU memory today: {today['peak_gpu_memory_gb']} GB.\")\nelse:\n    print(\"No baseline-fp16 row found in the uploaded leaderboard.csv - \"\n          \"nothing to compare against yet. Today's row is still logged above.\")",
    6: "def translate_with_llm(text, source_lang=\"Galician\", target_lang=\"English\"):\n    messages = [\n        {\n            \"role\": \"system\",\n            \"content\": (\n                f\"You are a professional translator. Translate the user's {source_lang} \"\n                f\"text into fluent {target_lang}. Output only the translation.\"\n            ),\n        },\n        {\"role\": \"user\", \"content\": text},\n    ]\n    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n    inputs = tokenizer(prompt, return_tensors=\"pt\").to(model.device)\n    with torch.no_grad():\n        output = model.generate(**inputs, max_new_tokens=200, do_sample=False)\n    generated = output[0][inputs[\"input_ids\"].shape[1]:]\n    return tokenizer.decode(generated, skip_special_tokens=True).strip()\n\n\ntracker_llm = EmissionsTracker(\n    project_name=\"green-mt-quantized-llm\",\n    measure_power_secs=1,\n    save_to_file=False,\n    log_level=\"warning\",\n)\ntracker_llm.start()\nstart_time = time.time()\ntranslation_llm_quantized = translate_with_llm(FULL_TEXT_GL)\nelapsed_llm = time.time() - start_time\nemissions_llm_kg = tracker_llm.stop()\n\nprint(f\"({elapsed_llm:.2f}s, {emissions_llm_kg * 1000:.4f} g CO2eq)\\n\")\nprint(translation_llm_quantized)",
    7: "import re\n\n\ndef parse_score(value):\n    try:\n        return float(value)\n    except (TypeError, ValueError):\n        match = re.search(r\"-?\\d+(\\.\\d+)?\", str(value))\n        return float(match.group()) if match else None\n\n\nbaseline_config = \"baseline-fp16\"\nquantized_config = f\"quantized-{QUANT_CONFIG_USED}\"\n\nif os.path.exists(\"mt_leaderboard.csv\"):\n    mtlb = pd.read_csv(\"mt_leaderboard.csv\")\n    baseline_mt = mtlb[mtlb[\"config\"] == baseline_config]\n    quantized_mt = mtlb[mtlb[\"config\"] == quantized_config]\n    if len(baseline_mt) > 0 and len(quantized_mt) > 0:\n        b_row = baseline_mt.iloc[-1]\n        q_row = quantized_mt.iloc[-1]\n        print(\"Translation cost, baseline-fp16 vs. quantized:\")\n        print(f\"  duration_sec: {b_row['duration_sec']:.2f} -> {q_row['duration_sec']:.2f}\")\n        print(f\"  emissions_g_co2eq: {b_row['emissions_kg_co2eq'] * 1000:.4f} -> \"\n              f\"{q_row['emissions_kg_co2eq'] * 1000:.4f}\")\n\n        fig, ax = plt.subplots(figsize=(5, 4))\n        ax.bar([baseline_config, quantized_config],\n               [b_row[\"emissions_kg_co2eq\"] * 1000, q_row[\"emissions_kg_co2eq\"] * 1000])\n        ax.set_ylabel(\"g CO2eq\")\n        ax.set_title(\"Translation cost: baseline vs. quantized\")\n        plt.tight_layout()\n        plt.show()\n    else:\n        print(\"mt_leaderboard.csv is missing a baseline-fp16 or quantized row --- \"\n              \"skipping the translation-cost comparison. Upload Session 1's \"\n              \"mt_leaderboard.csv before Step 10 to enable it.\")\nelse:\n    print(\"mt_leaderboard.csv not found --- skipping the translation-cost comparison.\")\n\nif os.path.exists(\"mt_eval_leaderboard.csv\"):\n    mtev = pd.read_csv(\"mt_eval_leaderboard.csv\")\n    mtev[\"parsed_score\"] = mtev[\"score\"].apply(parse_score)\n\n    metrics_present = sorted(mtev[\"metric\"].unique())\n    fig, axes = plt.subplots(1, len(metrics_present), figsize=(4 * len(metrics_present), 4))\n    if len(metrics_present) == 1:\n        axes = [axes]\n\n    print(\"\\nQuality scores, baseline general-llm vs. quantized general-llm-quantized:\")\n    for ax, metric in zip(axes, metrics_present):\n        sub = mtev[mtev[\"metric\"] == metric]\n        b = sub[sub[\"candidate\"] == \"general-llm\"]\n        q = sub[sub[\"candidate\"] == \"general-llm-quantized\"]\n        labels, values = [], []\n        if len(b) > 0:\n            labels.append(\"baseline\")\n            values.append(b.iloc[-1][\"parsed_score\"])\n        if len(q) > 0:\n            labels.append(\"quantized\")\n            values.append(q.iloc[-1][\"parsed_score\"])\n        ax.bar(labels, values)\n        ax.set_title(metric)\n        b_val = b.iloc[-1][\"parsed_score\"] if len(b) > 0 else None\n        q_val = q.iloc[-1][\"parsed_score\"] if len(q) > 0 else None\n        print(f\"  {metric}: baseline={b_val}, quantized={q_val}\")\n\n    plt.tight_layout()\n    plt.show()\nelse:\n    print(\"mt_eval_leaderboard.csv not found --- skipping the quality comparison.\")",
}
_fill_in = set(_solutions) if fill_in_solutions else set()


class ExerciseGaps(Exception):
    """A cell still has gaps marked ___."""


def _has_gap(lines):
    """True if `___` appears in the cell as code (not in a string or a comment)."""
    code = "".join(l for l in lines if not l.lstrip().startswith(("%", "!")))
    try:
        return any(t.type == tokenize.NAME and t.string == "___"
                   for t in tokenize.generate_tokens(io.StringIO(code).readline))
    except (tokenize.TokenError, SyntaxError):
        return False


def _fill_in_transformer(lines):
    """An exercise cell starts with `# Exercise N`; with fill-in on for N, it becomes the solution.
    A cell that still has gaps ___ stops with a message. (In IPython, ___ is a built-in name, the
    third-to-last output, so without this check the cell would run on with a wrong value.)"""
    m = re.match(r"# Exercise (\d+)\s*$", lines[0]) if lines else None
    if m and int(m.group(1)) in _fill_in:
        n = int(m.group(1))
        return [f"get_ipython().set_next_input(_solutions[{n}], replace=True)\n"] + (_solutions[n] + "\n").splitlines(True)
    if _has_gap(lines):
        return ['raise ExerciseGaps("this cell still has gaps marked ___: fill them in and run it again, '
                'or tick fill_in_solutions in the first code cell and run that cell")\n']
    return lines


_ip = get_ipython()
_ip.input_transformers_cleanup[:] = [t for t in _ip.input_transformers_cleanup
                                     if getattr(t, "__name__", "") != "_fill_in_transformer"]
_ip.input_transformers_cleanup.append(_fill_in_transformer)
if fill_in_solutions:
    print("✓ Every exercise cell you run from now on gets its solution.")

## Before running

You need a GPU runtime. In Colab: **Runtime -> Change runtime type -> T4 GPU**.

Expected outputs by the end:

- an updated `leaderboard.csv`;
- an updated `mt_leaderboard.csv`;
- an updated `mt_eval_leaderboard.csv`, with BLEU, BERTScore, and LLM-as-judge scores;
- printed comparison against Session 1 if the baseline files are present;
- a note on whether GPTQ Int4 or the fallback Int8 path loaded.


## Step 0 - install and import

Same libraries as Notebook 01a, plus quantization-related packages. Colab already includes `transformers`, `accelerate` and `pandas`; the cell installs only what is missing. `gptqmodel` (with `optimum`) runs the preferred GPTQ path, and `bitsandbytes` the fallback path. Building `gptqmodel` can take a few minutes; that is normal.


In [ ]:
!pip install -q bitsandbytes optimum codecarbon sacrebleu bert-score

import importlib.util
if importlib.util.find_spec("gptqmodel") is None:
    !pip install -q gptqmodel && echo "gptqmodel installed." || echo "gptqmodel failed to install. The GPTQ path will fall back to bitsandbytes int8."
else:
    print("gptqmodel already installed, skipping.")

**First time running this notebook in a session:** after the install cell above finishes, go to `Runtime` → `Restart session` once before continuing (this keeps the packages you just installed — it only clears the running Python process). Installing `gptqmodel` can pull in a different `numpy` build than what's already loaded, which otherwise shows up a few cells down as a confusing `numpy.ufunc` / `ModuleNotFoundError` when importing `transformers`. Restarting once avoids that; you don't need to re-run the install cell after restarting (if you do, it skips the build). Do run the first code cell ("Run this cell first") again: the restart cleared its gap check and its fill_in_solutions setting. Then continue from here.

In [ ]:
import time
import os
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GPTQConfig
from codecarbon import EmissionsTracker

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Total GPU memory (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("No GPU detected. This notebook is intended for a GPU runtime.")

import importlib.util

_missing_pkgs = [p for p in ("optimum", "bitsandbytes") if importlib.util.find_spec(p) is None]
if _missing_pkgs:
    raise ImportError(
        f"{', '.join(_missing_pkgs)} not importable in this Python process, even though "
        "the install cell above should have installed them. This almost always means the "
        "runtime needs a restart after installing new packages: go to Runtime > Restart "
        "session (NOT Disconnect/Delete runtime -- that would lose the install too), then "
        "re-run this notebook from the top. You do not need to re-run the pip install "
        "cell itself, it already worked; you just need a fresh process to pick it up."
    )

def gpu_memory_summary():
    if not torch.cuda.is_available():
        return {"peak_gpu_memory_gb": None, "current_gpu_memory_gb": None}
    return {
        "peak_gpu_memory_gb": round(torch.cuda.max_memory_allocated() / 1e9, 3),
        "current_gpu_memory_gb": round(torch.cuda.memory_allocated() / 1e9, 3),
    }

def get_tracker_energy_kwh(tracker_obj):
    try:
        return float(tracker_obj.final_emissions_data.energy_consumed)
    except (AttributeError, TypeError):
        return None

## Step 1 — The corpus

The exact same eight documents as Notebook 01a, unchanged — same task, same knowledge base.
The only thing changing today is the model's precision.


In [ ]:
CORPUS = {
    "training_vs_inference": """
Training a large language model is a one-time, very expensive event: the organization
that builds the model spends a large amount of compute once. Inference — actually using
the trained model to answer a query — is comparatively cheap per request, but it happens
every single time anyone uses the model. Once a model is deployed and used by many people,
the accumulated cost of inference tends to dominate the model's total lifetime footprint,
even though any single inference looks negligible next to the training run.
Source: Patterson et al., "Carbon Emissions and Large Neural Network Training," 2021
(arXiv:2104.10350). Training GPT-3 was estimated to emit 552 metric tons of CO2-equivalent,
using about 1,287 megawatt-hours of electricity.
""",
    "data_centers": """
Data centers are the physical facilities that house the servers running most of the
world's AI models. According to the International Energy Agency, global data center
electricity consumption was about 415 terawatt-hours in 2024, roughly 1.5 percent of all
electricity consumed worldwide that year. That demand is growing fast: electricity use by
data centers rose by about 17 percent in 2025 alone, driven mostly by AI workloads, and is
projected to approach 950 terawatt-hours by 2030, or about 3 percent of global electricity
demand.
Source: International Energy Agency, "Energy and AI" and related data centre electricity
reports, 2024-2025.
""",
    "quantization_basics": """
Neural network weights are normally stored as 32-bit floating point numbers (FP32).
Quantization reduces the number of bits used to represent each weight — for example to
16-bit (FP16/BF16), 8-bit integers (INT8), or even lower. Fewer bits per number means a
smaller model file and less memory traffic when running it, at the cost of some numerical
precision. A useful analogy: quantizing a model is like reducing the number of colors used
to represent an image — the picture is still recognizable, but some fine detail is lost.
Whether this actually saves energy in practice depends on the hardware: it helps most when
the bottleneck is moving data in and out of memory, not raw computation.
Source: Grootendorst, "A Visual Guide to Quantization," 2024.
""",
    "measuring_carbon": """
Estimating the carbon footprint of running a model requires several assumptions: how much
electricity the hardware draws, the Power Usage Effectiveness (PUE) of the data center
(extra power spent on cooling and overhead), and the carbon intensity of the local
electricity grid — how much CO2 is emitted per unit of electricity, which varies a lot by
region and by time of day. Because of all these assumptions, published carbon estimates can
vary widely, and some influential early estimates have later been revised — sometimes by an
order of magnitude — once other researchers re-examined the methodology. This is one reason
this course emphasizes measuring your own numbers rather than only citing others' headline
figures.
""",
    "inference_cost_task_type": """
Not all models cost the same to run for the same task. A 2024 study compared task-specific
models (built and trained for one narrow job) against multi-purpose generative models (like
general-purpose chatbots) performing the exact same tasks. For extractive question
answering, the most efficient task-specific models emitted around 0.3 grams of
CO2-equivalent per 1,000 inferences, while general-purpose generative models emitted around
10 grams of CO2-equivalent per 1,000 inferences for the same task — roughly 33 times more.
The gap was consistent across several other tasks too, even when controlling for the number
of parameters.
Source: Luccioni, Jernite & Strubell, "Power Hungry Processing: Watts Driving the Cost of AI
Deployment?", FAccT 2024 (arXiv:2311.16863).
""",
    "rag_concept": """
Retrieval-Augmented Generation, or RAG, is a technique where a model is given relevant
external text — retrieved from a document collection — as part of its input at query time,
instead of relying only on what it "learned" during training. This lets a smaller, cheaper
model answer questions about information it was never trained on, without the cost of
retraining or fine-tuning a much larger model on that information. The retrieval step
usually uses a separate, lightweight embedding model to find which documents are relevant
before the language model ever sees them.
""",
    "small_models_trend": """
Between 2024 and 2026, there has been a strong trend toward small, efficient open-weight
language models (often called SLMs, small language models) in the 1-4 billion parameter
range, designed to run on a single consumer GPU or even a laptop CPU after quantization.
These models are trained to be very capable for their size, and combined with quantization
and retrieval, they let many real applications avoid depending on a large, general-purpose
model at all — often at a fraction of the inference cost, if the task doesn't need the
breadth of a much bigger model.
""",
    "transformer_basics": """
Modern language models are built from transformer layers. Inside each layer,
"self-attention" lets every position in the input sequence gather information from every
other position, which is what allows the model to relate words like "it" back to the noun
it refers to several words earlier. This mechanism is powerful but has a computational cost
that grows faster than the length of the input (roughly quadratically with sequence
length), and more parameters mean more numbers to multiply and move through memory for
every single token generated. This is the underlying reason why both model size and input
length are direct levers on energy cost.
""",
}

for name, text in CORPUS.items():
    print(f"--- {name} ({len(text.split())} words) ---")
print(f"\nTotal documents: {len(CORPUS)}")


## Step 2 — The quiz

The exact same eight questions as Notebook 01a, checked the exact same way.


In [ ]:
QA_PAIRS = [
    {
        "question": "According to Patterson et al. (2021), how many metric tons of CO2-equivalent did training GPT-3 emit?",
        "reference": ["552"],
    },
    {
        "question": "Per the IEA, roughly what percentage of global electricity consumption did data centers represent in 2024?",
        "reference": ["1.5"],
    },
    {
        "question": "What does the acronym PUE stand for?",
        "reference": ["power usage effectiveness"],
    },
    {
        "question": "What does carbon footprint estimation depend on besides hardware power draw and grid carbon intensity?",
        "reference": ["pue", "power usage effectiveness"],
    },
    {
        "question": "In the 2024 study on inference costs, about how many grams of CO2-equivalent do the most efficient task-specific models emit per 1,000 extractive question-answering inferences?",
        "reference": ["0.3"],
    },
    {
        "question": "What is the name of the technique where a model is given retrieved external text at query time instead of relying only on what it learned during training?",
        "reference": ["retrieval-augmented generation", "rag"],
    },
    {
        "question": "What size range (in billions of parameters) do small language models (SLMs) typically fall into?",
        "reference": ["1-4", "1 to 4", "1-4 billion"],
    },
    {
        "question": "What mechanism inside a transformer layer lets every position gather information from every other position?",
        "reference": ["self-attention", "attention"],
    },
]

print(f"Number of questions: {len(QA_PAIRS)}")


## Step 3 — Load the quantized model

Two paths, tried in order.

Primary: `Qwen/Qwen2.5-1.5B-Instruct-GPTQ-Int4`, an official 4-bit checkpoint from the Qwen
team, already calibrated with GPTQ — we're loading a real pre-calibrated checkpoint, not
running calibration ourselves.

Fallback: if GPTQ's dependencies don't cooperate on this Colab image, load the plain
`Qwen/Qwen2.5-1.5B-Instruct` checkpoint with 8-bit quantization via `bitsandbytes` instead.
The same idea as Notebook 01e, where Approach 1 falls back to a saved answer when Apertium's
server is unreachable: try the more
specific real technique first, fall back to something more broadly reliable if the
environment can't handle it.

**Known limitation, worth noticing:** GPTQ's dependencies (`gptqmodel`/`optimum`) need a
CUDA build that matches the exact PyTorch version on the runtime, and that match can and
does break on a fresh Colab image. The fallback below exists because of a real, recurring
failure mode, not as a hypothetical.


In [ ]:
GPTQ_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct-GPTQ-Int4"
FALLBACK_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

loaded_model_name = None
loaded_config = None
load_started = time.time()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

try:
    print(f"Attempting primary path: {GPTQ_MODEL_NAME} (GPTQ, pre-calibrated 4-bit)")
    tokenizer = AutoTokenizer.from_pretrained(GPTQ_MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        GPTQ_MODEL_NAME,
        device_map="auto",
        quantization_config=GPTQConfig(bits=4, backend="torch"),
    )
    loaded_model_name = GPTQ_MODEL_NAME
    loaded_config = "gptq-int4"
except Exception as e:
    print("GPTQ path failed. Falling back to bitsandbytes 8-bit.")
    print("Reason:", repr(e))
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    tokenizer = AutoTokenizer.from_pretrained(FALLBACK_MODEL_NAME)
    model = AutoModelForCausalLM.from_pretrained(
        FALLBACK_MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        dtype=torch.float16,
    )
    loaded_model_name = FALLBACK_MODEL_NAME
    loaded_config = "bnb-int8-fallback"

model.eval()
load_seconds = time.time() - load_started
print("Loaded configuration:", loaded_config)
print("Loaded model:", loaded_model_name)
print("Load time (seconds):", round(load_seconds, 1))
print("GPU memory after load:", gpu_memory_summary())

MODEL_NAME = loaded_model_name
QUANT_CONFIG_USED = loaded_config


### ✏️ Exercise 1 — Count the bytes, and compare with the formula (advanced)

**Goal:** check the formula memory = parameters × bits ÷ 8 against the bytes the loaded model really holds.

The formula memory = parameters × bits ÷ 8 assumes every parameter is stored with the
same number of bits. Check it against the model you just loaded. Every tensor in PyTorch
knows its number of elements (`t.numel()`) and the bytes per element (`t.element_size()`).
A quantized model also keeps some of its tensors as **buffers** (for GPTQ: the packed 4-bit
weights, and the scales and zero points of each group), so count `model.buffers()` as well
as `model.parameters()`.

Complete `tensor_bytes` and `formula_gb`. The cell prints both, and the bytes per data
type.

**Reflection:** if the 4-bit model came out larger than the 4-bit formula, which tensors
take up the difference? Look at the list by data type: which parts of the model were not
quantized?

<details><summary><b>Hints</b> (click to open)</summary>

- `t.numel() * t.element_size()` is the size of one tensor in bytes.
- `list(m.parameters()) + list(m.buffers())` gives all the tensors.
- GB here means 10⁹ bytes (`1e9`), as in the Memory Estimates Playground (Notebook 02d).

</details>

In [ ]:
# Exercise 1
def tensor_bytes(m):
    # TODO: total bytes of all parameters and buffers of the model m
    ___


def formula_gb(n_params, bits):
    # TODO: parameters x bits / 8 bytes, in GB (10**9 bytes)
    ___

measured_gb = tensor_bytes(model) / 1e9
print(f"Loaded configuration: {loaded_config}")
print(f"Bytes in the model's tensors: {measured_gb:.2f} GB")
print(f"Formula, 1.54B parameters at 16 bits: {formula_gb(1.54e9, 16):.2f} GB")
print(f"Formula, 1.54B parameters at  4 bits: {formula_gb(1.54e9, 4):.2f} GB")
print(f"Formula, 1.54B parameters at  8 bits: {formula_gb(1.54e9, 8):.2f} GB")

by_dtype = {}
for t in list(model.parameters()) + list(model.buffers()):
    by_dtype[str(t.dtype)] = by_dtype.get(str(t.dtype), 0) + t.numel() * t.element_size()
for dtype, n in sorted(by_dtype.items(), key=lambda kv: -kv[1]):
    print(f"  {dtype:15s} {n / 1e9:.3f} GB")

In [ ]:
# Check for Exercise 1. Run this cell as is.
assert tensor_bytes(model) == model.get_memory_footprint(), \
    "count every parameter and buffer: numel() x element_size()"
assert abs(formula_gb(1.54e9, 16) - 3.080) < 1e-3
print("✓ Exercise 1 passed")

<details><summary><b>Solution to Exercise 1</b> (open it after you have tried)</summary>

```python
def tensor_bytes(m):
    tensors = list(m.parameters()) + list(m.buffers())
    return sum(t.numel() * t.element_size() for t in tensors)


def formula_gb(n_params, bits):
    return n_params * bits / 8 / 1e9

measured_gb = tensor_bytes(model) / 1e9
print(f"Loaded configuration: {loaded_config}")
print(f"Bytes in the model's tensors: {measured_gb:.2f} GB")
print(f"Formula, 1.54B parameters at 16 bits: {formula_gb(1.54e9, 16):.2f} GB")
print(f"Formula, 1.54B parameters at  4 bits: {formula_gb(1.54e9, 4):.2f} GB")
print(f"Formula, 1.54B parameters at  8 bits: {formula_gb(1.54e9, 8):.2f} GB")

by_dtype = {}
for t in list(model.parameters()) + list(model.buffers()):
    by_dtype[str(t.dtype)] = by_dtype.get(str(t.dtype), 0) + t.numel() * t.element_size()
for dtype, n in sorted(by_dtype.items(), key=lambda kv: -kv[1]):
    print(f"  {dtype:15s} {n / 1e9:.3f} GB")
```

</details>

<details><summary><b>Possible answers</b> (open it after you have thought about it)</summary>

Our run loaded the bitsandbytes int8 fallback (GPTQ did not install), so the GPTQ numbers below are an expectation from the sizes of the tensors stored in the checkpoint.

With GPTQ (4 bits), expect about 1.1–1.2 GB, against 0.77 GB for the 4-bit formula. Most of the difference is one table that stays in 16 bits: the vectors of the input pieces, 151,936 rows (one per piece) of 1,536 numbers, about 0.23 billion parameters or 0.47 GB. Qwen2.5-1.5B uses the same table again as its output table, the one that gives every piece its probability, so it is stored once. The scales and zero points of the groups add about 0.02 GB more. In the list by data type, look for a large `torch.float16` entry next to the packed `torch.int32` weights.

With the bitsandbytes int8 fallback, our run measured 1.78 GB against 1.54 GB for the 8-bit formula: 1.31 GB of `torch.int8` and 0.47 GB of `torch.float16`, the same table of input vectors, kept in 16 bits.

</details>

## Step 4 — The pipeline: same prompt, quantized model

Identical prompting logic to Notebook 01a — put all eight documents into the prompt, ask the
question. The only difference today is which model answers.


### ✏️ Exercise 2 — Generate greedily and keep only the answer

**Goal:** make every run repeatable (the model always writes the token with the highest probability) and keep only its output, without the prompt.

Complete `answer_question`:

1. Tokenize the prompt as PyTorch tensors on the model's device.
2. Generate at most `max_new_tokens` tokens with greedy decoding (always the most likely
   next token), so runs are repeatable.
3. Keep only the newly generated tokens.

<details><summary><b>Hints</b> (click to open)</summary>

- `tokenizer(text, return_tensors="pt")` returns PyTorch tensors; `.to(model.device)` moves them.
- `do_sample=False` means greedy decoding.
- `inputs["input_ids"].shape[1]` is the number of prompt tokens.

</details>

In [ ]:
# Exercise 2
def build_prompt(question, corpus):
    context = "\n\n".join(f"[{name}]\n{text.strip()}" for name, text in corpus.items())
    messages = [
        {
            "role": "system",
            "content": (
                "You are a concise assistant. Answer the question using ONLY the "
                "documents provided below. Answer in one short sentence."
            ),
        },
        {
            "role": "user",
            "content": f"Documents:\n{context}\n\nQuestion: {question}",
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def answer_question(question, corpus, max_new_tokens=60):
    prompt = build_prompt(question, corpus)
    inputs = ___  # TODO: tokenized prompt, PyTorch tensors, on model.device
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=___,
            do_sample=___,
        )
    generated = ___  # TODO: only the newly generated token ids
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


sample_answer = answer_question(QA_PAIRS[0]["question"], CORPUS)
print("Q:", QA_PAIRS[0]["question"])
print("A:", sample_answer)


In [ ]:
# Check for Exercise 2. Run this cell as is.
assert isinstance(sample_answer, str) and sample_answer, "empty answer"
assert "Documents:" not in sample_answer, "the answer still contains the prompt: slice off the prompt tokens"
_a = answer_question(QA_PAIRS[2]["question"], CORPUS, max_new_tokens=5)
assert len(tokenizer(_a)["input_ids"]) <= 6, "answer_question should respect max_new_tokens"
assert answer_question(QA_PAIRS[2]["question"], CORPUS, max_new_tokens=5) == _a, "greedy decoding is repeatable"
print("✓ Exercise 2 passed")

<details><summary><b>Solution to Exercise 2</b> (open it after you have tried)</summary>

```python
def build_prompt(question, corpus):
    context = "\n\n".join(f"[{name}]\n{text.strip()}" for name, text in corpus.items())
    messages = [
        {
            "role": "system",
            "content": (
                "You are a concise assistant. Answer the question using ONLY the "
                "documents provided below. Answer in one short sentence."
            ),
        },
        {
            "role": "user",
            "content": f"Documents:\n{context}\n\nQuestion: {question}",
        },
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)


def answer_question(question, corpus, max_new_tokens=60):
    prompt = build_prompt(question, corpus)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


sample_answer = answer_question(QA_PAIRS[0]["question"], CORPUS)
print("Q:", QA_PAIRS[0]["question"])
print("A:", sample_answer)
```

</details>

## Step 5 — Measure: run the full quiz with CodeCarbon

Same measurement setup as Notebook 01a: `EmissionsTracker` running the whole time.


In [ ]:
tracker = EmissionsTracker(
    project_name="green-nlp-quantized",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)

results = []
tracker.start()
start_time = time.time()

for pair in QA_PAIRS:
    answer = answer_question(pair["question"], CORPUS)
    results.append({"question": pair["question"], "reference": pair["reference"], "answer": answer})

elapsed = time.time() - start_time
emissions_kg = tracker.stop()

print(f"Ran {len(QA_PAIRS)} questions in {elapsed:.1f} seconds")
print(f"Estimated emissions: {emissions_kg * 1000:.4f} g CO2eq")


## Step 6 — Score quality

Same scoring rule as Notebook 01a: an answer counts as correct if it contains at least one
reference keyword.

**Known limitation, worth noticing:** the finding of Shi & Ding (2025), on the lesson screen "What measurements show", is
that smaller models are *more* vulnerable to quantization-induced quality loss than larger
ones, and ours is one of the smaller models out there. Don't be surprised either way — a
real accuracy drop here would be a genuine, expected finding, not a bug in this notebook.


In [ ]:
def score_answer(answer, reference_list):
    answer_lower = answer.lower()
    return any(ref.lower() in answer_lower for ref in reference_list)


for r in results:
    r["correct"] = score_answer(r["answer"], r["reference"])

accuracy = sum(r["correct"] for r in results) / len(results)

for r in results:
    mark = "✓" if r["correct"] else "✗"
    print(f"{mark} Q: {r['question']}")
    print(f"   A: {r['answer']}\n")

print(f"Accuracy: {accuracy:.0%} ({sum(r['correct'] for r in results)}/{len(results)})")


## Step 7 — Log this run to your leaderboard

Upload the `leaderboard.csv` you saved from Session 1 to this Colab session first (Files
sidebar, drag it in) — this cell reads whatever is there and adds today's row at the bottom.
If no file is found, it starts a fresh one instead of failing.


### ✏️ Exercise 3 — Append today's row to the leaderboard

**Goal:** record today's run in the same table as your Session 1 baseline, so the two can be compared row by row.

Fill in the last two fields of `leaderboard_row`, then add it as the last row of
the leaderboard you uploaded from Session 1.

<details><summary><b>Hints</b> (click to open)</summary>

- `mem` is a dict; its keys are printed by `gpu_memory_summary()`.
- `pd.DataFrame([row])` makes a one-row table from a dict; `pd.concat` stacks tables.

</details>

In [ ]:
# Exercise 3
import os

energy_kwh = get_tracker_energy_kwh(tracker)

mem = gpu_memory_summary()

leaderboard_row = {
    "config": loaded_config,
    "model": loaded_model_name,
    "params_billion": 1.5,
    "n_questions": len(QA_PAIRS),
    "duration_sec": round(elapsed, 2),
    "load_seconds": round(load_seconds, 2),
    "energy_kwh": energy_kwh,
    "emissions_kg_co2eq": emissions_kg,
    "accuracy": ___,
    "peak_gpu_memory_gb": ___,  # TODO: from mem
}

if os.path.exists("leaderboard.csv"):
    leaderboard = pd.read_csv("leaderboard.csv")
    leaderboard = ___  # TODO: the uploaded table with leaderboard_row added as the last row
    print(f"Appended to existing leaderboard.csv ({len(leaderboard) - 1} prior row(s)).")
else:
    leaderboard = pd.DataFrame([leaderboard_row])
    print("No existing leaderboard.csv found - starting a fresh one.")

leaderboard.to_csv("leaderboard.csv", index=False)
leaderboard


In [ ]:
# Check for Exercise 3. Run this cell as is.
assert leaderboard.iloc[-1]["config"] == loaded_config
assert leaderboard.iloc[-1]["accuracy"] == accuracy
assert len(pd.read_csv("leaderboard.csv")) == len(leaderboard)
print("✓ Exercise 3 passed")

<details><summary><b>Solution to Exercise 3</b> (open it after you have tried)</summary>

```python
import os

energy_kwh = get_tracker_energy_kwh(tracker)

mem = gpu_memory_summary()

leaderboard_row = {
    "config": loaded_config,
    "model": loaded_model_name,
    "params_billion": 1.5,
    "n_questions": len(QA_PAIRS),
    "duration_sec": round(elapsed, 2),
    "load_seconds": round(load_seconds, 2),
    "energy_kwh": energy_kwh,
    "emissions_kg_co2eq": emissions_kg,
    "accuracy": accuracy,
    "peak_gpu_memory_gb": mem["peak_gpu_memory_gb"],
}

if os.path.exists("leaderboard.csv"):
    leaderboard = pd.read_csv("leaderboard.csv")
    leaderboard = pd.concat([leaderboard, pd.DataFrame([leaderboard_row])], ignore_index=True)
    print(f"Appended to existing leaderboard.csv ({len(leaderboard) - 1} prior row(s)).")
else:
    leaderboard = pd.DataFrame([leaderboard_row])
    print("No existing leaderboard.csv found - starting a fresh one.")

leaderboard.to_csv("leaderboard.csv", index=False)
leaderboard
```

</details>

### ✏️ Exercise 4 — Joules per token and per correct answer (advanced)

**Goal:** compare runs by energy per unit of useful work, a number that does not depend on where the electricity came from.

Grams of CO2eq depend on where the electricity comes from. Energy per unit of
useful work compares runs more directly. Report today's run in three such units: joules
per generated token, per question, and per **correct** answer. 1 kWh = 3,600,000 J.

Complete `count_tokens`, `joules` and `joules_per_correct`. If the leaderboard has the
Session 1 baseline row, the cell compares the two runs per question and per correct
answer.

**Reflection:** a cheaper model that answers fewer questions correctly can cost more per
correct answer. Is that the case here?

<details><summary><b>Hints</b> (click to open)</summary>

- `tokenizer(text)["input_ids"]` is the list of token ids.
- Multiply kWh by 3.6e6.
- `sum(r["correct"] for r in results_list)` counts the correct answers.

</details>

In [ ]:
# Exercise 4
def count_tokens(text):
    # TODO: the number of tokens in text, with the model's tokenizer
    ___


def joules(kwh):
    # TODO: convert kWh to joules
    ___


def joules_per_correct(total_joules, results_list):
    # TODO: energy divided by the number of correct answers (infinity if there are none)
    ___

n_generated = sum(count_tokens(r["answer"]) for r in results)
today = joules(energy_kwh)
print(f"Today ({loaded_config}): {today:.1f} J for {n_generated} generated tokens")
print(f"  {today / n_generated:.3f} J per generated token")
print(f"  {today / len(results):.1f} J per question")
print(f"  {joules_per_correct(today, results):.1f} J per correct answer")

baseline_rows = leaderboard[leaderboard["config"] == "baseline-fp16"]
if len(baseline_rows) > 0 and pd.notna(baseline_rows.iloc[-1]["energy_kwh"]):
    b = baseline_rows.iloc[-1]
    b_joules = joules(b["energy_kwh"])
    print(f"Baseline-fp16 (Session 1): {b_joules / b['n_questions']:.1f} J per question, "
          f"{b_joules / max(b['accuracy'] * b['n_questions'], 1e-12):.1f} J per correct answer")

In [ ]:
# Check for Exercise 4. Run this cell as is.
assert joules(1) == 3.6e6
assert joules_per_correct(100.0, [{"correct": True}, {"correct": False}, {"correct": True}]) == 50.0
assert joules_per_correct(100.0, [{"correct": False}]) == float("inf")
assert count_tokens("Power Usage Effectiveness") == len(tokenizer("Power Usage Effectiveness")["input_ids"])
print("✓ Exercise 4 passed")

<details><summary><b>Solution to Exercise 4</b> (open it after you have tried)</summary>

```python
def count_tokens(text):
    return len(tokenizer(text)["input_ids"])


def joules(kwh):
    return kwh * 3.6e6


def joules_per_correct(total_joules, results_list):
    n_correct = sum(r["correct"] for r in results_list)
    return total_joules / n_correct if n_correct else float("inf")

n_generated = sum(count_tokens(r["answer"]) for r in results)
today = joules(energy_kwh)
print(f"Today ({loaded_config}): {today:.1f} J for {n_generated} generated tokens")
print(f"  {today / n_generated:.3f} J per generated token")
print(f"  {today / len(results):.1f} J per question")
print(f"  {joules_per_correct(today, results):.1f} J per correct answer")

baseline_rows = leaderboard[leaderboard["config"] == "baseline-fp16"]
if len(baseline_rows) > 0 and pd.notna(baseline_rows.iloc[-1]["energy_kwh"]):
    b = baseline_rows.iloc[-1]
    b_joules = joules(b["energy_kwh"])
    print(f"Baseline-fp16 (Session 1): {b_joules / b['n_questions']:.1f} J per question, "
          f"{b_joules / max(b['accuracy'] * b['n_questions'], 1e-12):.1f} J per correct answer")
```

</details>

<details><summary><b>Possible answers</b> (open it after you have thought about it)</summary>

Energy per correct answer is energy per question divided by the share of correct answers. Compare the two printed lines. If today's accuracy is the same as the baseline's, the two units tell the same story: whichever run used less energy per question also used less per correct answer. If today's accuracy dropped, today's cost per correct answer rises by that factor: with 6 of 8 right instead of 8 of 8, it is a third higher than its cost per question would suggest. Quantization then pays off per correct answer only if it saved more energy than it lost in accuracy. If no baseline row was uploaded, answer for today's run alone: how much more does a correct answer cost than a question?

</details>

### Where this row fits

If a `baseline-fp16` row is already in the leaderboard, here's how today's numbers compare
directly.


### ✏️ Exercise 5 — How much cheaper was today's run?

**Goal:** reduce the comparison with the baseline to one number: how many times less (or more) CO2eq today's run emitted.

Compute `ratio`: the baseline's emissions divided by today's. A value above 1 means
today's run was cheaper. Guard against dividing by zero.

<details><summary><b>Hints</b> (click to open)</summary>

- `baseline` and `today` are rows of the leaderboard; the column is `emissions_kg_co2eq`.
- `max(x, 1e-12)` is never 0.

</details>

In [ ]:
# Exercise 5
import matplotlib.pyplot as plt

baseline_rows = leaderboard[leaderboard["config"] == "baseline-fp16"]

if len(baseline_rows) > 0:
    baseline = baseline_rows.iloc[-1]
    today = leaderboard.iloc[-1]

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

    configs = [baseline["config"], today["config"]]
    energies = [baseline["emissions_kg_co2eq"] * 1000, today["emissions_kg_co2eq"] * 1000]
    accuracies = [baseline["accuracy"] * 100, today["accuracy"] * 100]

    axes[0].bar(configs, energies)
    axes[0].set_ylabel("g CO2eq")
    axes[0].set_title("Emissions")
    axes[0].tick_params(axis="x", rotation=15)

    axes[1].bar(configs, accuracies)
    axes[1].set_ylabel("accuracy (%)")
    axes[1].set_title("Accuracy")
    axes[1].set_ylim(0, 100)
    axes[1].tick_params(axis="x", rotation=15)

    plt.tight_layout()
    plt.show()

    ratio = ___  # TODO: baseline emissions / today's emissions, never dividing by 0
    print(f"baseline-fp16 emitted {ratio:.2f}x as much CO2eq as today's run "
          f"(>1 means today's run was cheaper).")
    print(f"Accuracy: baseline {baseline['accuracy']:.0%} vs. today {today['accuracy']:.0%}.")
    if "peak_gpu_memory_gb" in today and pd.notna(today["peak_gpu_memory_gb"]):
        print(f"Peak GPU memory today: {today['peak_gpu_memory_gb']} GB.")
else:
    print("No baseline-fp16 row found in the uploaded leaderboard.csv - "
          "nothing to compare against yet. Today's row is still logged above.")


In [ ]:
# Check for Exercise 5. Run this cell as is.
if len(baseline_rows) > 0:
    assert abs(ratio - baseline["emissions_kg_co2eq"] / today["emissions_kg_co2eq"]) < 1e-9, ratio
else:
    print("No baseline-fp16 row was uploaded, so there is no ratio to check.")
print("✓ Exercise 5 passed")

<details><summary><b>Solution to Exercise 5</b> (open it after you have tried)</summary>

```python
import matplotlib.pyplot as plt

baseline_rows = leaderboard[leaderboard["config"] == "baseline-fp16"]

if len(baseline_rows) > 0:
    baseline = baseline_rows.iloc[-1]
    today = leaderboard.iloc[-1]

    fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))

    configs = [baseline["config"], today["config"]]
    energies = [baseline["emissions_kg_co2eq"] * 1000, today["emissions_kg_co2eq"] * 1000]
    accuracies = [baseline["accuracy"] * 100, today["accuracy"] * 100]

    axes[0].bar(configs, energies)
    axes[0].set_ylabel("g CO2eq")
    axes[0].set_title("Emissions")
    axes[0].tick_params(axis="x", rotation=15)

    axes[1].bar(configs, accuracies)
    axes[1].set_ylabel("accuracy (%)")
    axes[1].set_title("Accuracy")
    axes[1].set_ylim(0, 100)
    axes[1].tick_params(axis="x", rotation=15)

    plt.tight_layout()
    plt.show()

    ratio = baseline["emissions_kg_co2eq"] / max(today["emissions_kg_co2eq"], 1e-12)
    print(f"baseline-fp16 emitted {ratio:.2f}x as much CO2eq as today's run "
          f"(>1 means today's run was cheaper).")
    print(f"Accuracy: baseline {baseline['accuracy']:.0%} vs. today {today['accuracy']:.0%}.")
    if "peak_gpu_memory_gb" in today and pd.notna(today["peak_gpu_memory_gb"]):
        print(f"Peak GPU memory today: {today['peak_gpu_memory_gb']} GB.")
else:
    print("No baseline-fp16 row found in the uploaded leaderboard.csv - "
          "nothing to compare against yet. Today's row is still logged above.")
```

</details>

In [ ]:
import os

if os.path.exists("leaderboard.csv"):
    print("Saved leaderboard.csv in this Colab session.")
    print("To download it: click the folder icon in the left sidebar, find")
    print("leaderboard.csv, right-click it, and choose Download.")
else:
    print("leaderboard.csv not found — re-run the cell above first.")


## Part 2 — Machine translation, quantized (required)

Notebook 01e translated our Galician passage four ways; the general-purpose LLM was
one of them. Here we redo just that one approach — same passage, same prompt, the
quantized model instead of the fp16 one — and measure it the same way.


## Step 8 — The text

The same fragment and full passage as Notebook 01e.


In [ ]:
FRAGMENT_GL = "unha tía miña"

FULL_TEXT_GL = (
    "Ás veces é mellor estar calado. Unha tía miña, que todo o solucionaba con ditos e "
    "refráns, dicía algo así como \"un é escravo do que di e dono do que cala\". Sempre me "
    "pareceu un sabio aforismo, e se cadra debería terlle feito caso nalgunhas ocasións, "
    "aquelas que, aínda que raras, contribuíron a dar bruscos cambios de sentido á miña vida."
)

print(f"Fragment: \"{FRAGMENT_GL}\"")
print(f"Full text ({len(FULL_TEXT_GL.split())} words):\n{FULL_TEXT_GL}")


## Step 9 — Translate with the quantized model

Same prompt template as Notebook 01e's general-purpose-LLM approach.


### ✏️ Exercise 6 — Measure the translation

**Goal:** measure the cost of one translation by the quantized model, the same way Notebook 01e measured the fp16 one.

Measure one translation: start the tracker and a clock, translate `FULL_TEXT_GL`,
then store the translation, the elapsed seconds and the emissions.

<details><summary><b>Hints</b> (click to open)</summary>

- The same pattern as Step 5: `tracker.start()`, `time.time()`, ..., `tracker.stop()`.
- `tracker_llm.stop()` returns the emissions in kg CO2eq.

</details>

In [ ]:
# Exercise 6
def translate_with_llm(text, source_lang="Galician", target_lang="English"):
    messages = [
        {
            "role": "system",
            "content": (
                f"You are a professional translator. Translate the user's {source_lang} "
                f"text into fluent {target_lang}. Output only the translation."
            ),
        },
        {"role": "user", "content": text},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


tracker_llm = EmissionsTracker(
    project_name="green-mt-quantized-llm",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
# TODO: start tracker_llm and a clock, translate FULL_TEXT_GL with translate_with_llm,
# then store translation_llm_quantized, elapsed_llm (seconds) and emissions_llm_kg
___

print(f"({elapsed_llm:.2f}s, {emissions_llm_kg * 1000:.4f} g CO2eq)\n")
print(translation_llm_quantized)


In [ ]:
# Check for Exercise 6. Run this cell as is.
assert isinstance(translation_llm_quantized, str) and len(translation_llm_quantized.split()) > 10
assert 0 < elapsed_llm < 3600, elapsed_llm
assert emissions_llm_kg is not None and emissions_llm_kg > 0
print("✓ Exercise 6 passed")

<details><summary><b>Solution to Exercise 6</b> (open it after you have tried)</summary>

```python
def translate_with_llm(text, source_lang="Galician", target_lang="English"):
    messages = [
        {
            "role": "system",
            "content": (
                f"You are a professional translator. Translate the user's {source_lang} "
                f"text into fluent {target_lang}. Output only the translation."
            ),
        },
        {"role": "user", "content": text},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=200, do_sample=False)
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


tracker_llm = EmissionsTracker(
    project_name="green-mt-quantized-llm",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker_llm.start()
start_time = time.time()
translation_llm_quantized = translate_with_llm(FULL_TEXT_GL)
elapsed_llm = time.time() - start_time
emissions_llm_kg = tracker_llm.stop()

print(f"({elapsed_llm:.2f}s, {emissions_llm_kg * 1000:.4f} g CO2eq)\n")
print(translation_llm_quantized)
```

</details>

### Comparing against Session 1

Look back at what you printed for the general-purpose LLM's translation in Notebook 01e —
same passage, same model family, only the precision changed. Does the wording hold up?
Does it still flatten the aunt into "my aunt" instead of introducing her as a new
character, or does something change?


## Step 10 — Log this comparison

Same upload-and-append pattern as Step 7, for `mt_leaderboard.csv` this time: upload the one you
saved from Notebook 01e before you run the next cell.


In [ ]:
mt_leaderboard_row = {
    "task": "translation",
    "era": "general-llm-quantized",
    "config": f"quantized-{loaded_config}",
    "model": loaded_model_name,
    "duration_sec": elapsed_llm,
    "energy_kwh": get_tracker_energy_kwh(tracker_llm),
    "emissions_kg_co2eq": emissions_llm_kg,
    "translation": translation_llm_quantized,
}

if os.path.exists("mt_leaderboard.csv"):
    mt_leaderboard = pd.read_csv("mt_leaderboard.csv")
    mt_leaderboard = pd.concat([mt_leaderboard, pd.DataFrame([mt_leaderboard_row])], ignore_index=True)
    print(f"Appended to existing mt_leaderboard.csv ({len(mt_leaderboard) - 1} prior row(s)).")
else:
    mt_leaderboard = pd.DataFrame([mt_leaderboard_row])
    print("No existing mt_leaderboard.csv found - starting a fresh one.")

mt_leaderboard.to_csv("mt_leaderboard.csv", index=False)
mt_leaderboard


In [ ]:
if os.path.exists("mt_leaderboard.csv"):
    print("Saved mt_leaderboard.csv in this Colab session.")
    print("To download it: click the folder icon in the left sidebar, find")
    print("mt_leaderboard.csv, right-click it, and choose Download.")
else:
    print("mt_leaderboard.csv not found — re-run the cell above first.")


In [ ]:
REFERENCE_EN = (
    "Sometimes it's better to stay quiet. I had an aunt who solved everything with sayings "
    "and proverbs; she used to say something like, 'You're a slave to what you say, and "
    "master of what you keep to yourself.' It always struck me as a wise saying, and maybe "
    "I should have listened to it on a few occasions — the rare ones that, even so, ended up "
    "sharply changing the course of my life."
)


## Step 11 — BLEU (2002): count overlapping n-grams

Same first metric as Notebook 01e's Step 10, same idea: count how many word sequences
(n-grams) the candidate translation shares with the reference. No model, no GPU --- pure
counting.


In [ ]:
import sacrebleu

candidates = {"general-llm-quantized": translation_llm_quantized}

tracker_bleu = EmissionsTracker(
    project_name="green-eval-bleu-quantized",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker_bleu.start()
start_time = time.time()

bleu_scores = {
    name: sacrebleu.corpus_bleu([text], [[REFERENCE_EN]]).score
    for name, text in candidates.items()
}

elapsed_bleu = time.time() - start_time
emissions_bleu_kg = tracker_bleu.stop()

for name, score in bleu_scores.items():
    print(f"{name}: BLEU = {score:.2f}")
print(f"\n({elapsed_bleu:.4f}s, {emissions_bleu_kg * 1000:.6f} g CO2eq)")
print("\nStep 16 compares this with the general-llm row of your Session 1 mt_eval_leaderboard.csv, "
      "same reference, same metric, only the model's precision changed.")


## Step 12 — BERTScore (2020): a learned, neural metric

Same fail-soft pattern as Notebook 01e's Step 11: BERTScore compares candidate and reference
using contextual embeddings instead of counting exact overlaps, so it can reward a correct
paraphrase that BLEU would penalize. Same smaller `distilbert-base-uncased` model for
speed, for the same reason: the biggest model isn't needed to see the point.


In [ ]:
bertscore_available = False
bertscore_scores = {}
elapsed_bertscore = 0.0
emissions_bertscore_kg = 0.0

try:
    from bert_score import score as bertscore

    tracker_bertscore = EmissionsTracker(
        project_name="green-eval-bertscore-quantized",
        measure_power_secs=1,
        save_to_file=False,
        log_level="warning",
    )
    tracker_bertscore.start()
    start_time = time.time()

    cand_list = list(candidates.values())
    ref_list = [REFERENCE_EN] * len(cand_list)
    _, _, f1 = bertscore(
        cand_list, ref_list, model_type="distilbert-base-uncased", verbose=False
    )
    bertscore_scores = dict(zip(candidates.keys(), f1.tolist()))

    elapsed_bertscore = time.time() - start_time
    emissions_bertscore_kg = tracker_bertscore.stop()
    bertscore_available = True

    for name, s in bertscore_scores.items():
        print(f"{name}: BERTScore F1 = {s:.4f}")
    print(f"\n({elapsed_bertscore:.2f}s, {emissions_bertscore_kg * 1000:.4f} g CO2eq)")
except Exception as e:
    print(f"BERTScore failed to load/run in this environment ({type(e).__name__}: {e}).")
    print("Continuing without a BERTScore score for this run.")


## Step 13 — LLM-as-a-judge: prompt a model to grade the translation

Same approach as Notebook 01e's Step 12. The quantized model already loaded today grades
its own translation --- no new download.


In [ ]:
def llm_judge_score(source, candidate, reference):
    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert translation evaluator. Given a source text, a candidate "
                "translation, and a reference translation, rate the candidate's quality "
                "from 1 (very poor) to 10 (excellent), considering both accuracy and "
                "fluency. Output ONLY the number."
            ),
        },
        {
            "role": "user",
            "content": (
                f"Source (Galician): {source}\n\n"
                f"Reference translation: {reference}\n\n"
                f"Candidate translation: {candidate}\n\n"
                "Score (1-10):"
            ),
        },
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=10, do_sample=False)
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()


tracker_judge = EmissionsTracker(
    project_name="green-eval-llm-judge-quantized",
    measure_power_secs=1,
    save_to_file=False,
    log_level="warning",
)
tracker_judge.start()
start_time = time.time()

judge_scores = {
    name: llm_judge_score(FULL_TEXT_GL, text, REFERENCE_EN)
    for name, text in candidates.items()
}

elapsed_judge = time.time() - start_time
emissions_judge_kg = tracker_judge.stop()

for name, score in judge_scores.items():
    print(f"{name}: LLM-judge score = {score}")
print(f"\n({elapsed_judge:.2f}s, {emissions_judge_kg * 1000:.4f} g CO2eq)")


### A second, independent judge: ChatGPT, by hand

Same reasoning as Notebook 01e: the score above came from the same quantized Qwen model
already loaded in this notebook grading its own output --- a known weak spot of
LLM-as-a-judge. To check that, get a second opinion from a genuinely independent model:
ChatGPT (or Claude, Gemini, whichever you have access to), run by hand, outside this
notebook.

The cell below prints a ready-to-paste prompt for today's translation --- the exact same
prompt Qwen was just given. Paste it into your chat tool of choice, then fill the number
it returns into `CHATGPT_JUDGE_SCORES` in the cell after that.


In [ ]:
def build_judge_prompt(source, candidate, reference):
    return (
        "You are an expert translation evaluator. Given a source text, a candidate "
        "translation, and a reference translation, rate the candidate's quality from 1 "
        "(very poor) to 10 (excellent), considering both accuracy and fluency. Output "
        "ONLY the number.\n\n"
        f"Source (Galician): {source}\n\n"
        f"Reference translation: {reference}\n\n"
        f"Candidate translation: {candidate}\n\n"
        "Score (1-10):"
    )


for name, text in candidates.items():
    print(f"===== prompt for: {name} =====")
    print(build_judge_prompt(FULL_TEXT_GL, text, REFERENCE_EN))
    print()


Replace `None` below with the number your chat tool returned.
No chat tool at hand? Leave it as `None`: the notebook then runs on without this judge.


In [ ]:
CHATGPT_JUDGE_SCORES = {
    "general-llm-quantized": None,
}

chatgpt_judge_scores = CHATGPT_JUDGE_SCORES
chatgpt_available = all(v is not None for v in chatgpt_judge_scores.values())
if chatgpt_available:
    for name, score in chatgpt_judge_scores.items():
        print(f"{name}: LLM-judge score (ChatGPT, by hand) = {score}")
else:
    print("Still None in CHATGPT_JUDGE_SCORES: the rest of the notebook runs without the ChatGPT judge.")
    print("To add it, fill in the score above, then run this cell and the cells after it again.")

elapsed_chatgpt_judge = None
emissions_chatgpt_judge_kg = None


## Step 14 — Compare the four evaluation methods

Same idea as Notebook 01e's Step 13: one candidate this time, four ways of scoring it ---
three with a measurable local cost, one (the ChatGPT judge) that has no local cost by
design, since the actual grading ran outside this notebook.


In [ ]:
eval_rows = [
    {"metric": "BLEU (2002)", "needs_gpu": False,
     "duration_sec": round(elapsed_bleu, 4),
     "emissions_g_co2eq": round(emissions_bleu_kg * 1000, 6),
     **{f"{k}_score": round(v, 2) for k, v in bleu_scores.items()}},
]
if bertscore_available:
    eval_rows.append({"metric": "BERTScore (2020)", "needs_gpu": True,
     "duration_sec": round(elapsed_bertscore, 2),
     "emissions_g_co2eq": round(emissions_bertscore_kg * 1000, 4),
     **{f"{k}_score": round(v, 4) for k, v in bertscore_scores.items()}})
eval_rows.append({"metric": "LLM-as-judge (Qwen, today)", "needs_gpu": True,
     "duration_sec": round(elapsed_judge, 2),
     "emissions_g_co2eq": round(emissions_judge_kg * 1000, 4),
     **{f"{k}_score": v for k, v in judge_scores.items()}})
if chatgpt_available:
    eval_rows.append({"metric": "LLM-as-judge (ChatGPT, by hand)", "needs_gpu": False,
     "duration_sec": elapsed_chatgpt_judge,
     "emissions_g_co2eq": emissions_chatgpt_judge_kg,
     **{f"{k}_score": v for k, v in chatgpt_judge_scores.items()}})

eval_comparison = pd.DataFrame(eval_rows)
print(eval_comparison.to_string(index=False))

cheapest_eval = eval_comparison.loc[eval_comparison["emissions_g_co2eq"].idxmin()]
priciest_eval = eval_comparison.loc[eval_comparison["emissions_g_co2eq"].idxmax()]
eval_ratio = priciest_eval["emissions_g_co2eq"] / max(cheapest_eval["emissions_g_co2eq"], 1e-12)

plt.figure(figsize=(7, 4))
plt.bar(eval_comparison["metric"], eval_comparison["emissions_g_co2eq"])
plt.yscale("log")
plt.ylabel("g CO2eq (log scale)")
plt.title(f"Grading cost today: ~{eval_ratio:,.0f}x apart, cheapest to priciest")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()
if chatgpt_available:
    print("(The ChatGPT judge has no bar above --- its cost was never local to this notebook")
    print("to measure, not zero.)")


## Step 15 — Log the evaluation comparison

Same upload-and-append pattern as Step 7 and Step 10, for `mt_eval_leaderboard.csv` this
time. Upload the file you saved from Session 1 first, the same way, so this appends
instead of starting over.


In [ ]:
leaderboard_rows = []
for name, score in bleu_scores.items():
    leaderboard_rows.append({
        "metric": "bleu", "candidate": name, "score": round(score, 2),
        "duration_sec": elapsed_bleu,
        "energy_kwh": get_tracker_energy_kwh(tracker_bleu), "emissions_kg_co2eq": emissions_bleu_kg,
    })
if bertscore_available:
    for name, score in bertscore_scores.items():
        leaderboard_rows.append({
            "metric": "bertscore", "candidate": name, "score": round(score, 4),
            "duration_sec": elapsed_bertscore,
            "energy_kwh": get_tracker_energy_kwh(tracker_bertscore), "emissions_kg_co2eq": emissions_bertscore_kg,
        })
for name, score in judge_scores.items():
    leaderboard_rows.append({
        "metric": "llm_judge_qwen", "candidate": name, "score": score,
        "duration_sec": elapsed_judge,
        "energy_kwh": get_tracker_energy_kwh(tracker_judge), "emissions_kg_co2eq": emissions_judge_kg,
    })
if chatgpt_available:
    for name, score in chatgpt_judge_scores.items():
        leaderboard_rows.append({
            "metric": "llm_judge_chatgpt", "candidate": name, "score": score,
            "duration_sec": elapsed_chatgpt_judge,
            "energy_kwh": None, "emissions_kg_co2eq": emissions_chatgpt_judge_kg,
        })

if os.path.exists("mt_eval_leaderboard.csv"):
    mt_eval_leaderboard = pd.read_csv("mt_eval_leaderboard.csv")
    mt_eval_leaderboard = pd.concat(
        [mt_eval_leaderboard, pd.DataFrame(leaderboard_rows)], ignore_index=True
    )
    print(f"Appended to existing mt_eval_leaderboard.csv ({len(mt_eval_leaderboard) - len(leaderboard_rows)} prior row(s)).")
else:
    mt_eval_leaderboard = pd.DataFrame(leaderboard_rows)
    print("No existing mt_eval_leaderboard.csv found - starting a fresh one.")

mt_eval_leaderboard.to_csv("mt_eval_leaderboard.csv", index=False)
mt_eval_leaderboard


In [ ]:
if os.path.exists("mt_eval_leaderboard.csv"):
    print("Saved mt_eval_leaderboard.csv in this Colab session.")
    print("To download it: click the folder icon in the left sidebar, find")
    print("mt_eval_leaderboard.csv, right-click it, and choose Download.")
else:
    print("mt_eval_leaderboard.csv not found --- re-run the cell above first.")


## Step 16 — Compare against your Session 1 baseline

The actual quality-tradeoff question: not just whether the quantized translation grades
reasonably on its own, but whether it grades differently from the fp16 general-purpose LLM
measured in Session 1 --- on cost (`mt_leaderboard.csv`) and on quality
(`mt_eval_leaderboard.csv`) at once. This only works if both files were uploaded before
Step 10 and Step 15 above; if either is missing, this cell says so and skips that half of
the comparison instead of failing.


### ✏️ Exercise 7 — Find the baseline and quantized rows

**Goal:** pick the fp16 and the quantized translation out of the logs, so their cost and scores can stand side by side.

Select the rows of `mtlb` whose `config` is `baseline_config`, and those whose
`config` is `quantized_config`.

<details><summary><b>Hints</b> (click to open)</summary>

- `table[table["col"] == value]` keeps the rows where the condition is True.

</details>

In [ ]:
# Exercise 7
import re


def parse_score(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        match = re.search(r"-?\d+(\.\d+)?", str(value))
        return float(match.group()) if match else None


baseline_config = "baseline-fp16"
quantized_config = f"quantized-{QUANT_CONFIG_USED}"

if os.path.exists("mt_leaderboard.csv"):
    mtlb = pd.read_csv("mt_leaderboard.csv")
    baseline_mt = ___  # TODO: rows of mtlb for baseline_config
    quantized_mt = ___  # TODO: rows of mtlb for quantized_config
    if len(baseline_mt) > 0 and len(quantized_mt) > 0:
        b_row = baseline_mt.iloc[-1]
        q_row = quantized_mt.iloc[-1]
        print("Translation cost, baseline-fp16 vs. quantized:")
        print(f"  duration_sec: {b_row['duration_sec']:.2f} -> {q_row['duration_sec']:.2f}")
        print(f"  emissions_g_co2eq: {b_row['emissions_kg_co2eq'] * 1000:.4f} -> "
              f"{q_row['emissions_kg_co2eq'] * 1000:.4f}")

        fig, ax = plt.subplots(figsize=(5, 4))
        ax.bar([baseline_config, quantized_config],
               [b_row["emissions_kg_co2eq"] * 1000, q_row["emissions_kg_co2eq"] * 1000])
        ax.set_ylabel("g CO2eq")
        ax.set_title("Translation cost: baseline vs. quantized")
        plt.tight_layout()
        plt.show()
    else:
        print("mt_leaderboard.csv is missing a baseline-fp16 or quantized row --- "
              "skipping the translation-cost comparison. Upload Session 1's "
              "mt_leaderboard.csv before Step 10 to enable it.")
else:
    print("mt_leaderboard.csv not found --- skipping the translation-cost comparison.")

if os.path.exists("mt_eval_leaderboard.csv"):
    mtev = pd.read_csv("mt_eval_leaderboard.csv")
    mtev["parsed_score"] = mtev["score"].apply(parse_score)

    metrics_present = sorted(mtev["metric"].unique())
    fig, axes = plt.subplots(1, len(metrics_present), figsize=(4 * len(metrics_present), 4))
    if len(metrics_present) == 1:
        axes = [axes]

    print("\nQuality scores, baseline general-llm vs. quantized general-llm-quantized:")
    for ax, metric in zip(axes, metrics_present):
        sub = mtev[mtev["metric"] == metric]
        b = sub[sub["candidate"] == "general-llm"]
        q = sub[sub["candidate"] == "general-llm-quantized"]
        labels, values = [], []
        if len(b) > 0:
            labels.append("baseline")
            values.append(b.iloc[-1]["parsed_score"])
        if len(q) > 0:
            labels.append("quantized")
            values.append(q.iloc[-1]["parsed_score"])
        ax.bar(labels, values)
        ax.set_title(metric)
        b_val = b.iloc[-1]["parsed_score"] if len(b) > 0 else None
        q_val = q.iloc[-1]["parsed_score"] if len(q) > 0 else None
        print(f"  {metric}: baseline={b_val}, quantized={q_val}")

    plt.tight_layout()
    plt.show()
else:
    print("mt_eval_leaderboard.csv not found --- skipping the quality comparison.")


In [ ]:
# Check for Exercise 7. Run this cell as is.
if "quantized_mt" in globals():
    assert len(quantized_mt) > 0 and (quantized_mt["config"] == quantized_config).all()
    assert (baseline_mt["config"] == baseline_config).all()
else:
    print("mt_leaderboard.csv was not found, so there is nothing to check.")
print("✓ Exercise 7 passed")

<details><summary><b>Solution to Exercise 7</b> (open it after you have tried)</summary>

```python
import re


def parse_score(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        match = re.search(r"-?\d+(\.\d+)?", str(value))
        return float(match.group()) if match else None


baseline_config = "baseline-fp16"
quantized_config = f"quantized-{QUANT_CONFIG_USED}"

if os.path.exists("mt_leaderboard.csv"):
    mtlb = pd.read_csv("mt_leaderboard.csv")
    baseline_mt = mtlb[mtlb["config"] == baseline_config]
    quantized_mt = mtlb[mtlb["config"] == quantized_config]
    if len(baseline_mt) > 0 and len(quantized_mt) > 0:
        b_row = baseline_mt.iloc[-1]
        q_row = quantized_mt.iloc[-1]
        print("Translation cost, baseline-fp16 vs. quantized:")
        print(f"  duration_sec: {b_row['duration_sec']:.2f} -> {q_row['duration_sec']:.2f}")
        print(f"  emissions_g_co2eq: {b_row['emissions_kg_co2eq'] * 1000:.4f} -> "
              f"{q_row['emissions_kg_co2eq'] * 1000:.4f}")

        fig, ax = plt.subplots(figsize=(5, 4))
        ax.bar([baseline_config, quantized_config],
               [b_row["emissions_kg_co2eq"] * 1000, q_row["emissions_kg_co2eq"] * 1000])
        ax.set_ylabel("g CO2eq")
        ax.set_title("Translation cost: baseline vs. quantized")
        plt.tight_layout()
        plt.show()
    else:
        print("mt_leaderboard.csv is missing a baseline-fp16 or quantized row --- "
              "skipping the translation-cost comparison. Upload Session 1's "
              "mt_leaderboard.csv before Step 10 to enable it.")
else:
    print("mt_leaderboard.csv not found --- skipping the translation-cost comparison.")

if os.path.exists("mt_eval_leaderboard.csv"):
    mtev = pd.read_csv("mt_eval_leaderboard.csv")
    mtev["parsed_score"] = mtev["score"].apply(parse_score)

    metrics_present = sorted(mtev["metric"].unique())
    fig, axes = plt.subplots(1, len(metrics_present), figsize=(4 * len(metrics_present), 4))
    if len(metrics_present) == 1:
        axes = [axes]

    print("\nQuality scores, baseline general-llm vs. quantized general-llm-quantized:")
    for ax, metric in zip(axes, metrics_present):
        sub = mtev[mtev["metric"] == metric]
        b = sub[sub["candidate"] == "general-llm"]
        q = sub[sub["candidate"] == "general-llm-quantized"]
        labels, values = [], []
        if len(b) > 0:
            labels.append("baseline")
            values.append(b.iloc[-1]["parsed_score"])
        if len(q) > 0:
            labels.append("quantized")
            values.append(q.iloc[-1]["parsed_score"])
        ax.bar(labels, values)
        ax.set_title(metric)
        b_val = b.iloc[-1]["parsed_score"] if len(b) > 0 else None
        q_val = q.iloc[-1]["parsed_score"] if len(q) > 0 else None
        print(f"  {metric}: baseline={b_val}, quantized={q_val}")

    plt.tight_layout()
    plt.show()
else:
    print("mt_eval_leaderboard.csv not found --- skipping the quality comparison.")
```

</details>

## Wrap-up: everything we measured today

Every measurement of today in one unit, g CO2eq. Exercise 5 and Step 16 compared them with Session 1.


In [ ]:
summary_rows = [
    {"item": "QA quiz, 8 questions (Step 5)", "g_co2eq": emissions_kg * 1000},
    {"item": "translation (Step 9)", "g_co2eq": emissions_llm_kg * 1000},
    {"item": "BLEU (Step 11)", "g_co2eq": emissions_bleu_kg * 1000},
]
if bertscore_available:
    summary_rows.append({"item": "BERTScore (Step 12)", "g_co2eq": emissions_bertscore_kg * 1000})
summary_rows.append({"item": "LLM judge, Qwen (Step 13)", "g_co2eq": emissions_judge_kg * 1000})

summary = pd.DataFrame(summary_rows)
print(f"Today's runs ({QUANT_CONFIG_USED}):")
print(summary.to_string(index=False))

plt.figure(figsize=(7, 4))
plt.bar(summary["item"], summary["g_co2eq"])
plt.yscale("log")
plt.ylabel("g CO2eq (log scale)")
plt.title("Everything measured today")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

**Key takeaways**

- Quantization changed exactly one thing about the pipeline — how many bits the model's
  numbers use — and today's leaderboard rows show whether that actually saved energy for
  *this* model, on *this* hardware, not just whether it's supposed to in theory.
- Which path loaded, GPTQ or the bitsandbytes fallback, is itself a real data point: GPTQ's
  dependencies are genuinely more fragile to install than plain `transformers`, the same
  kind of tooling friction Notebook 01e met with Apertium's server and BERTScore.
- Accuracy and energy don't have to move together. A real drop in accuracy here is not a
  failure of this notebook — smaller models are measurably more vulnerable to quantization
  damage than larger ones, and 1.5B parameters is small.
- The translation comparison isolates the same lever in a second task: same passage, same
  model family, only precision changed. Whatever held up or broke in the QA numbers can
  show up differently, or not at all, in translation quality.
- Evaluation has its own cost too, as in Notebook 01e's Part 2: BLEU is nearly free,
  BERTScore runs a small neural model, and the LLM judge runs a whole generative model;
  compare their `energy_kwh` rows in `mt_eval_leaderboard.csv`. Step 16 puts the quantized
  model next to your fp16 baseline, on cost and on quality: that comparison, not an
  assumption, answers the question this session set out to measure.


## Final worksheet - interpret today's run

Fill these in before the share-out.

1. Which configuration loaded: GPTQ Int4 or bitsandbytes Int8 fallback?
2. QA accuracy: baseline vs. today. Did it change?
3. CO2eq or energy: baseline vs. today. Did it change?
4. Time for the eight questions (`duration_sec`): baseline vs. today. Did it change?
5. Peak GPU memory today. (Session 1 did not record it, so there is no baseline value; Notebook
   02c measures both models.)
6. MT output: did quantization change anything important in the translation, especially around `unha tía miña`?
7. BLEU, BERTScore, and LLM-judge: did they agree on direction (better/worse/unchanged) versus your Session 1 baseline? If not, which would you trust for this one passage?
8. Would you accept this quantized model for this specific task? Why or why not?


## Step 17 — Share-out (discussion)

Before we wrap up, compare notes with your neighbor.

**On the QA leaderboard:**

- Which path loaded for you, GPTQ or the bitsandbytes fallback? Did that differ from your
  neighbor's, and if so, why might that be?
- How did your accuracy compare to your Session 1 baseline? If it dropped, look at which
  questions changed from correct to wrong — anything in common?
- How did energy and emissions compare to baseline-fp16? Is the direction (cheaper or more
  expensive) what you expected going in?

**On the translation comparison:**

- Read the quantized translation next to what you remember or saved from Notebook 01e's
  fp16 version. Same wording, or different? Does either one introduce the aunt as a new
  character, or does both flatten her into "my aunt"?
- Look at Step 16's comparison: did BLEU, BERTScore, and the LLM judge all move in the
  same direction, or did they disagree? If they disagreed, which would you trust, and why?
  What would you actually need to trust any of these comparisons between two
  translations of one short passage — is one number, from one metric, ever enough?


## What you learned: check yourself

Answer each question from what you ran and read above. If one is hard, go back to the
step it comes from.

1. Which quantized configuration loaded in your runtime? Why does the notebook have a fallback at all?
2. Between your Session 1 baseline and today's run only the precision changed. Which of accuracy, energy
   and time moved, and in which direction?
3. Why would a drop in accuracy here be an expected finding, not a bug, for a model of 1.5B parameters?
4. Which model graded the quantized translation in Step 13, and why does the notebook ask for a second judge
   from outside it?

## Step 18 — Code literacy check

Before you're done, pick at least two or three code cells above that you actually ran and
read today and add `#` comments to them explaining, in your own words, what each meaningful
line or block does and *why* it's there, in language a classmate who hasn't seen the code
could follow.

Do this part on your own, after class:

1. Paste your commented code into an LLM chat (ChatGPT, Claude, Gemini — whichever you
   have access to).
2. Ask it something like: *"Here is code I commented myself, explaining what I think it
   does. Based only on my comments, how well do you think I understood this code? What
   did I get right, what did I miss, and what's still fuzzy?"*
3. Read what it says. Where it disagrees with your comment, go check the code again —
   who's right?

This is a self-check on your own understanding: a way to catch, in your own words, where
your mental model of the code breaks down against a second reader.



## Optional extension, if time remains

Run the QA section twice with the same quantized configuration. The answers should repeat (greedy decoding, as Exercise 1 checks), but time and energy
readings vary from run to run. If your two rows differ, do not treat that as a mistake; treat it as a reminder that one measurement is a sample, not a law.
